# AOI defect classification — MobileNetV2 walkthrough

The same steps as the original Colab notebook, but every step is a call into `src/aoi_detection`.
Run from the repo root after `pip install -e ".[train]"`.

> Training 30 epochs on CPU takes a while; lower `epochs` in the config cell for a quick check.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
import os; os.chdir(ROOT)

from aoi_detection.config import load_train_config
cfg = load_train_config('configs/train_mobilenet_v2.yaml')
cfg

## 1. Image information

In [ ]:
from aoi_detection.data.dataset import load_labels, class_counts, CLASS_NAMES
from aoi_detection.utils.visualize import plot_image_info, plot_class_samples, plot_class_distribution

train_list = load_labels(cfg.train_csv_path)
plot_image_info(cfg.train_images_dir / train_list.loc[0, 'ID'], show=True)

## 2. One sample per class

In [ ]:
plot_class_samples(train_list, cfg.train_images_dir, show=True)

## 3. Image numbers per class

In [ ]:
print(class_counts(train_list))
plot_class_distribution(train_list, show=True)

## 4. Train / valid split (8:2) and class weights

In [ ]:
from aoi_detection.data.dataset import split_train_valid, compute_class_weights

train, valid = split_train_valid(train_list, cfg.valid_split, cfg.seed)
print(len(train), len(valid))
print(class_counts(train).to_dict())
compute_class_weights(train)

## 5. Train MobileNetV2

`train()` builds the generators (with augmentation), the model, checkpoints the best `val_accuracy` epoch and saves `accuracy.png` / `loss.png` into the run directory.

In [ ]:
import dataclasses
from aoi_detection.training.trainer import train

# quick_cfg = dataclasses.replace(cfg, epochs=3)   # uncomment for a smoke test
result = train(cfg, run_name='notebook')
result.run_dir, result.best_val_accuracy, result.best_epoch

## 6. Accuracy and loss curves

In [ ]:
from aoi_detection.utils.visualize import plot_history
plot_history(result.history, show=True)

## 7. Evaluate on test.csv, write submission, confusion matrix

In [ ]:
from aoi_detection.training.evaluate import evaluate

ev = evaluate(cfg, result.run_dir, show=True)
print('accuracy:', ev.accuracy)
print(ev.report)
ev.predictions.head()